# Week 2 — Session 5
## Query Performance, Storage Layout, and Data Formats

## Q1 — Full Table Scans

Without an index, the database may need to check every row in the `orders` table.

This is called a **full table scan**.

As the table becomes larger, more rows must be checked, so the query usually becomes slower.

One way to improve the query is to create an index on `customer_id`.

## Q2 — Indexing Basics

I would create an index on `customer_id`.

The index can help the database locate matching rows more directly instead of checking the entire table.

One disadvantage is that indexes use additional storage and add extra work when rows are inserted, updated, or deleted.

## Q3 — Observe a Scan and an Index

This exercise compares the query plan before and after creating an index on `customer_id`.

In [1]:
import sqlite3

connection = sqlite3.connect(":memory:")
cursor = connection.cursor()

cursor.execute("""
CREATE TABLE orders (
    order_id INTEGER PRIMARY KEY,
    customer_id INTEGER NOT NULL,
    amount_gbp REAL NOT NULL,
    status TEXT NOT NULL
)
""")

orders = [
    (
        order_id,
        (order_id % 500) + 1,
        round(10 + (order_id % 90), 2),
        "Dispatched" if order_id % 2 == 0 else "Confirmed"
    )
    for order_id in range(1, 5001)
]

cursor.executemany(
    "INSERT INTO orders VALUES (?, ?, ?, ?)",
    orders
)

connection.commit()

query = """
SELECT order_id, amount_gbp
FROM orders
WHERE customer_id = 101
"""

cursor.execute(f"EXPLAIN QUERY PLAN {query}")

print("Before index:")
for row in cursor.fetchall():
    print(row)

cursor.execute("""
CREATE INDEX idx_orders_customer_id
ON orders(customer_id)
""")

cursor.execute(f"EXPLAIN QUERY PLAN {query}")

print("\nAfter index:")
for row in cursor.fetchall():
    print(row)

connection.close()

Before index:
(2, 0, 216, 'SCAN orders')

After index:
(3, 0, 61, 'SEARCH orders USING INDEX idx_orders_customer_id (customer_id=?)')


## Q4 — Filter Early

This exercise compares:

- reading all rows and filtering in Python
- filtering and aggregating directly in SQL

Filtering in SQL usually avoids transferring unnecessary rows.

In [2]:
import sqlite3

connection = sqlite3.connect(":memory:")
cursor = connection.cursor()

cursor.execute("""
CREATE TABLE orders (
    order_id INTEGER PRIMARY KEY,
    amount_gbp REAL NOT NULL,
    status TEXT NOT NULL
)
""")

orders = [
    (1, 34.99, "Dispatched"),
    (2, 18.50, "Confirmed"),
    (3, 42.75, "Dispatched"),
    (4, 12.00, "Cancelled")
]

cursor.executemany(
    "INSERT INTO orders VALUES (?, ?, ?)",
    orders
)

# Approach A
cursor.execute("SELECT * FROM orders")
all_orders = cursor.fetchall()

python_total = sum(
    amount
    for _, amount, status in all_orders
    if status == "Dispatched"
)

# Approach B
cursor.execute("""
SELECT SUM(amount_gbp)
FROM orders
WHERE status = 'Dispatched'
""")

sql_total = cursor.fetchone()[0]

print(f"Python total: £{python_total:.2f}")
print(f"SQL total: £{sql_total:.2f}")
print(f"Totals match: {python_total == sql_total}")

connection.close()

Python total: £77.74
SQL total: £77.74
Totals match: True


## Q5–Q9 — Multiple Choice

Q5 — B  
Q6 — A  
Q7 — B  
Q8 — C  
Q9 — A

## Q10 — Partitioning Fundamentals

A suitable partition key is `order_date`.

Because most reports filter by date, I would partition the data by year and month.

Example:

```text
/orders/year=2026/month=01/
/orders/year=2026/month=02/

Partition pruning allows the query engine to read only the relevant partitions instead of scanning all years and months.


The workbook itself uses year/month partitioning as the suggested pattern for date-filtered reports. :contentReference[oaicite:2]{index=2}

---

## Q11 — Markdown cell

```markdown
## Q11 — Design a Partition Scheme

I would partition the data by year and month.

Example:

```text
/orders/year=2026/month=01/
/orders/year=2026/month=02/

This is a good choice because reports usually filter by month.

Partitioning by hour would create too many small partitions for a query pattern that rarely filters by hour.


---

## Q12 — Markdown cell

```markdown
## Q12 — Small-Files Problem

Having thousands of very small files can reduce performance because the system spends extra time opening files and managing metadata.

A common solution is **compaction**, where many small files are combined into fewer larger files.

## Q13 — Combine Small CSV Files

This exercise combines several small CSV files into one larger CSV file.

In [4]:
from pathlib import Path
import csv

input_folder = Path("small_order_files")
input_folder.mkdir(exist_ok=True)

files_and_rows = {
    "orders_1.csv": [1001, "Amelia Clarke", 34.99],
    "orders_2.csv": [1002, "Oliver Bennett", 18.50],
    "orders_3.csv": [1003, "Aisha Khan", 42.75]
}

header = ["order_id", "customer_name", "amount_gbp"]

for file_name, row in files_and_rows.items():
    with (input_folder / file_name).open(
        "w",
        newline="",
        encoding="utf-8"
    ) as file:
        writer = csv.writer(file)
        writer.writerow(header)
        writer.writerow(row)

output_file = Path("orders_combined.csv")

combined_rows = []

for file_path in sorted(input_folder.glob("*.csv")):
    with file_path.open(
        "r",
        newline="",
        encoding="utf-8"
    ) as file:
        reader = csv.DictReader(file)
        combined_rows.extend(reader)

with output_file.open(
    "w",
    newline="",
    encoding="utf-8"
) as file:
    writer = csv.DictWriter(file, fieldnames=header)
    writer.writeheader()
    writer.writerows(combined_rows)

print("Combined rows:", len(combined_rows))

Combined rows: 3


## Q14–Q18 — Multiple Choice

Q14 — B  
Q15 — B  
Q16 — A  
Q17 — B  
Q18 — B

## Q19 — Compare File Formats

| Format | Type | Orientation | Common Use |
|---|---|---|---|
| CSV | Text | Row-like | Simple exchange and human-readable data |
| Parquet | Binary | Columnar | Analytics and data lakes |
| Avro | Binary | Row-oriented | Structured event exchange and streaming |
| ORC | Binary | Columnar | Large analytical workloads, often Hadoop/Hive |

## Q19 — Compare File Formats

| Format | Type | Orientation | Common Use |
|---|---|---|---|
| CSV | Text | Row-like | Simple exchange and human-readable data |
| Parquet | Binary | Columnar | Analytics and data lakes |
| Avro | Binary | Row-oriented | Structured event exchange and streaming |
| ORC | Binary | Columnar | Large analytical workloads, often Hadoop/Hive |

## Q20 — Row vs Columnar Storage

CSV stores complete rows.

For analytical queries that need only a few columns, the system may still need to process more data than necessary.

A columnar format such as Parquet stores values from the same columns together.

This makes it easier for analytical engines to read only the required columns.

## Q21 — Choose a File Format

Small manually opened file → CSV

Large analytical dataset → Parquet

Structured streaming events → Avro

Large Hive analytical tables → ORC

## Q22 — Avro-Style Schema

This example creates a JSON representation of an Avro-style schema.

The optional `coupon_code` field uses `null` as an allowed type and has a default value.

In [5]:
import json

schema = {
    "type": "record",
    "name": "OrderEvent",
    "fields": [
        {"name": "order_id", "type": "string"},
        {"name": "customer_id", "type": "string"},
        {"name": "amount_gbp", "type": "double"},
        {"name": "order_status", "type": "string"},
        {
            "name": "coupon_code",
            "type": ["null", "string"],
            "default": None
        }
    ]
}

with open(
    "order_event_schema.json",
    "w",
    encoding="utf-8"
) as file:
    json.dump(schema, file, indent=2)

with open(
    "order_event_schema.json",
    "r",
    encoding="utf-8"
) as file:
    loaded_schema = json.load(file)

field_names = [
    field["name"]
    for field in loaded_schema["fields"]
]

print(field_names)

['order_id', 'customer_id', 'amount_gbp', 'order_status', 'coupon_code']


## Q23–Q27 — Multiple Choice

Q23 — B  
Q24 — B  
Q25 — B  
Q26 — A  
Q27 — B

## Q28 — Identify the Requirements

### Best partition field

`order_date`, partitioned by year and month.

### Suitable analytical format

Parquet or ORC.

### Required columns

- order_date
- product_id
- quantity
- amount

### Why avoid SELECT *

`SELECT *` reads columns that are not needed.

Selecting only required columns can reduce unnecessary I/O and processing.

## Q29 — Storage Decision Table

| Dataset | Format | Partition | Reason |
|---|---|---|---|
| Raw order exports | CSV | ingestion date | Keeps original source files |
| Clean analytical orders | Parquet | order year/month | Efficient analytical queries |
| Streaming click events | Avro | event date/hour | Good for structured event data |
| Monthly sales summaries | Parquet or ORC | year/month | Fast analytical reading |

## Q30 — Create a Partitioned Folder Layout

This exercise creates folders by year and month and writes each order into the correct partition.

In [6]:
from pathlib import Path
from datetime import datetime
import csv

orders = [
    {
        "order_id": 1001,
        "order_date": "2026-06-28",
        "amount_gbp": 34.99
    },
    {
        "order_id": 1002,
        "order_date": "2026-07-02",
        "amount_gbp": 18.50
    },
    {
        "order_id": 1003,
        "order_date": "2026-07-17",
        "amount_gbp": 42.75
    }
]

base_folder = Path("orders_lake")

header = [
    "order_id",
    "order_date",
    "amount_gbp"
]

for order in orders:

    order_date = datetime.strptime(
        order["order_date"],
        "%Y-%m-%d"
    )

    partition_folder = (
        base_folder
        / f"year={order_date.year}"
        / f"month={order_date.month:02d}"
    )

    partition_folder.mkdir(
        parents=True,
        exist_ok=True
    )

    output_file = partition_folder / "orders.csv"

    file_exists = output_file.exists()

    with output_file.open(
        "a",
        newline="",
        encoding="utf-8"
    ) as file:

        writer = csv.DictWriter(
            file,
            fieldnames=header
        )

        if not file_exists:
            writer.writeheader()

        writer.writerow(order)

for file_path in sorted(
    base_folder.rglob("orders.csv")
):
    print(file_path)

orders_lake/year=2026/month=06/orders.csv
orders_lake/year=2026/month=07/orders.csv


## Q31 — Improve the Analytics Design

I would make these improvements:

1. Convert analytical data from CSV to Parquet.
2. Partition the data by commonly filtered dates.
3. Select only required columns instead of using `SELECT *`.
4. Compact many small files into fewer larger files.
5. Use compression where appropriate.

These changes can reduce unnecessary data reads and improve analytical performance.

## Q32–Q36 — Multiple Choice

Q32 — A  
Q33 — B  
Q34 — A  
Q35 — A  
Q36 — B